In [ ]:
!pip -q install vllm 2>&1 | tail -1
!pip -q install transformers accelerate librosa soundfile 2>&1 | tail -1

In [ ]:
import os,glob
os.makedirs('/kaggle/working/lifemate',exist_ok=True)
os.chdir('/kaggle/working')
print(glob.glob('/kaggle/input/*/*')[:10])

In [ ]:
%%writefile /kaggle/working/lifemate/__init__.py

# package


In [ ]:
%%writefile /kaggle/working/lifemate/agent.py
"""Baseline LifeMate agent: transcript -> LLM (JSON action per step) -> ToolEnv. Batched across samples (step-synchronous)."""
import json, re, time
from .common import add_zaic, Retriever

SYSTEM = """You are a Vietnamese voice-assistant agent. You get the TRANSCRIPT of the user's latest spoken request (it may contain speech-recognition mistakes) plus context and the tools of the installed apps. You act ONLY by calling tools, one tool per step.

Reply with exactly ONE JSON object and nothing else:
{"action":"call","name":"<app.tool>","arguments":{...}}
{"action":"ask","tool_name":"<app.tool>","params":["<missing_param>", ...]}   (required info is missing and cannot be found with read tools; this ends the episode)
{"action":"finish"}   (the request is fully done; or NO installed app can handle it - then call nothing at all)

Rules:
- Use only tools listed below; never invent tools or parameters. Follow each tool's parameter names, types and enums exactly.
- Never guess an id. Look ids up with read tools first (contacts.search, *.list_*, *.get_status ...) and copy them exactly from tool results or earlier turns.
- Do read calls before write calls. Do not repeat a call. Do nothing the user did not ask for.
- If the user cancels or negates ("thôi", "khỏi", "đừng"), or no installed app fits the request, reply {"action":"finish"} immediately without any call.
- If a required parameter cannot be determined from the request, context or tool results, use the "ask" action (name the tool you wanted and the missing parameter names).
- Dates/times: ISO 8601 with +07:00, computed from current_time ("mai" = tomorrow, "ba giờ chiều" = 15:00, "5 rưỡi" = 05:30). Numbers: convert spoken numbers ("hai triệu tám" = 2800000, "năm trăm nghìn" = 500000).
- Free-text fields (title, content, label, name, note ...): keep the user's own words.
- If the user corrects themselves ("5 giờ, à không, 5 rưỡi"), use the final value."""

def tool_line(t):
    props = (t.get('parameters') or {}).get('properties', {})
    req = set((t.get('parameters') or {}).get('required', []))
    ps = []
    for k, v in props.items():
        s = f"{k}:{v.get('type', 'any')}"
        if 'enum' in v: s += '=' + '|'.join(map(str, v['enum']))
        if k in req: s += '*'
        if v.get('x-ref'): s += '(id)'
        ps.append(s)
    oneof = (t.get('parameters') or {}).get('x-one-of')
    extra = f" ONE-OF{json.dumps(oneof, ensure_ascii=False)}" if oneof else ''
    return f"- {t['name']} [{t['kind']}] {t.get('description', '')} | params: {', '.join(ps) or '-'}{extra}"

def tools_for(item, catalog_by_app):
    lines = []
    for app in item['context']['installed_apps']:
        for t in catalog_by_app.get(app, []): lines.append(tool_line(t))
    return '\n'.join(lines)

def short(x, n=1500):
    s = json.dumps(x, ensure_ascii=False); return s if len(s) <= n else s[:n] + '…'

def render_history(item, texts):
    out = []
    for i, h in enumerate(item.get('history') or []):
        if h['role'] == 'user':
            out.append('User: ' + texts.get(h['audio'], h.get('audio_text', '')))
        else:
            s = 'Assistant: ' + h.get('text', '')
            for c, r in zip(h.get('tool_calls') or [], h.get('tool_results') or []):
                s += f"\n  [tool] {c['name']} {json.dumps(c['arguments'], ensure_ascii=False)} -> {short(r, 600)}"
            out.append(s)
    return '\n'.join(out) or '(none)'

def parse_action(text):
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S).strip()
    m = re.search(r'\{.*\}', text, flags=re.S)
    if not m: return None
    s = m.group(0)
    for cand in (s, s[:s.rfind('}') + 1]):
        try: return json.loads(cand)
        except Exception: continue
    return None

class Run:
    def __init__(self, item, zaic, root, split, catalog_by_app, utter, texts, shots_text, max_steps=8):
        self.item, self.split, self.max_steps = item, split, max_steps
        self.env = zaic.ToolEnv(split, item['id'], root=root)
        ctx = item['context']; prof = ctx.get('user_profile', {})
        user = (f"current_time: {ctx['current_time']} ({ctx['timezone']})\nlocation: {ctx.get('location')}\nuser_profile: {json.dumps(prof, ensure_ascii=False)}\n"
                f"installed apps: {', '.join(ctx['installed_apps'])}\n\nTOOLS:\n{tools_for(item, catalog_by_app)}\n"
                f"- sys.ask_missing_info [meta] use via the \"ask\" action\n\nCONVERSATION SO FAR:\n{render_history(item, texts)}\n\n"
                f"CURRENT REQUEST (transcript): {utter}")
        sysmsg = SYSTEM + ('\n\n' + shots_text if shots_text else '')
        self.messages = [{'role': 'system', 'content': sysmsg}, {'role': 'user', 'content': user}]
        self.done, self.steps, self.bad, self.seen, self.t0 = False, 0, 0, set(), time.time()
        self.log = []
    def feed(self, text):
        self.steps += 1; self.messages.append({'role': 'assistant', 'content': text})
        act = parse_action(text)
        if not isinstance(act, dict) or act.get('action') not in ('call', 'ask', 'finish'):
            self.bad += 1; self._tool('Invalid output. Reply with exactly one JSON action object.')
            if self.bad >= 2: self.done = True
            return
        a = act['action']
        if a == 'finish': self.done = True; return
        if a == 'ask':
            tn, ps = act.get('tool_name'), act.get('params') or []
            self.env.call('sys.ask_missing_info', {'tool_name': tn, 'params': ps}); self.done = True; return
        name, args = act.get('name'), act.get('arguments') or {}
        key = (name, json.dumps(args, sort_keys=True, ensure_ascii=False))
        if key in self.seen:
            self._tool('Duplicate call ignored. Continue with the next step or finish.'); self.bad += 1
            if self.bad >= 3: self.done = True
            return
        v = self.env.validate(name, args)
        if not v.get('ok', False):
            self._tool('REJECTED (not executed): ' + str(v.get('error'))); self.bad += 1
            if self.bad >= 3: self.done = True
            return
        self.seen.add(key); res = self.env.call(name, args); self._tool(short(res))
        self.log.append((name, args, res.get('ok')))
    def _tool(self, content): self.messages.append({'role': 'user', 'content': 'TOOL RESULT: ' + content})
    def result(self): return {'tool_calls': self.env.get_tool_calls(), 'state_hash': self.env.get_state_hash(), 'seconds': time.time() - self.t0, 'steps': self.steps}

def build_shots(item, train_rows, retr, k, exclude_ids=()):
    if not k: return ''
    q = item.get('_text', '')
    idx = [i for i in retr.top(q, k + len(exclude_ids)) if train_rows[i]['id'] not in exclude_ids][:k]
    parts = ['EXAMPLES from similar past requests (ids below are illustrative: ALWAYS look real ids up with tools). Format: request -> ordered calls.']
    for i in idx:
        r = train_rows[i]; cs = r['label']['tool_calls']
        seq = '; '.join(f"{c['name']} {json.dumps(c['arguments'], ensure_ascii=False)}" for c in cs) or '(no calls: out of scope / cancelled)'
        parts.append(f"- \"{r['audio_text']}\" -> {seq}")
    return '\n'.join(parts)

def run_batch(items, root, split, catalog, backend, texts_of, train_rows=None, retr=None, shots=0, exclude=None, max_steps=8, log=print):
    """items: sample dicts; texts_of(item)->(utterance, {audio_path: text}). Returns {id: result}."""
    zaic = add_zaic(root, split)
    cba = {a['id']: a['tools'] for a in catalog['apps']}
    runs = []
    for it in items:
        utt, texts = texts_of(it); it['_text'] = utt
        ex = {it['id']} if (exclude is None or exclude) and shots else set()
        runs.append(Run(it, zaic, root, split, cba, utt, texts, build_shots(it, train_rows, retr, shots, ex) if shots else '', max_steps))
    for step in range(max_steps):
        act = [r for r in runs if not r.done and r.steps < max_steps]
        if not act: break
        t0 = time.time(); outs = backend.generate([r.messages for r in act])
        for r, o in zip(act, outs): r.feed(o)
        log(f'step {step}: {len(act)} active, {time.time() - t0:.1f}s')
    return {r.item['id']: r.result() for r in runs}


In [ ]:
%%writefile /kaggle/working/lifemate/asr.py
"""Stage 1: transcribe every wav (final turn + history user turns) with Whisper; cache to JSON {relative_audio_path: text}."""
import json, sys, time
from pathlib import Path

def collect(root, split, rows, final_only=False):
    paths = []
    for it in rows:
        paths.append(it['audio'])
        if not final_only:
            for h in it.get('history') or []:
                if h['role'] == 'user': paths.append(h['audio'])
    return sorted(set(paths))

def transcribe(root, split, paths, model='openai/whisper-large-v3-turbo', bs=16, out=None):
    import torch, soundfile as sf
    from transformers import pipeline
    pipe = pipeline('automatic-speech-recognition', model=model, torch_dtype=torch.float16, device=0, chunk_length_s=30)
    res = json.load(open(out, encoding='utf-8')) if out and Path(out).exists() else {}
    todo = [p for p in paths if p not in res]; t0 = time.time()
    for i in range(0, len(todo), bs):
        chunk = todo[i:i + bs]
        arrs = [sf.read(str(Path(root) / split / p))[0].astype('float32') for p in chunk]
        outs = pipe([{'raw': a, 'sampling_rate': 16000} for a in arrs], batch_size=bs, generate_kwargs={'language': 'vi', 'task': 'transcribe', 'num_beams': 1})
        for p, o in zip(chunk, outs): res[p] = o['text'].strip()
        if out and (i // bs) % 10 == 0:
            json.dump(res, open(out, 'w', encoding='utf-8'), ensure_ascii=False); print(f'asr {len(res)}/{len(paths)} {time.time() - t0:.0f}s', flush=True)
    if out: json.dump(res, open(out, 'w', encoding='utf-8'), ensure_ascii=False)
    return res


In [ ]:
%%writefile /kaggle/working/lifemate/backends.py
"""LLM backends. 'vllm' (fast, batched) or 'hf' (transformers, slower), plus 'stub' for dry runs."""
class Stub:
    def __init__(self, reply='{"action":"finish"}'): self.reply = reply
    def generate(self, batch): return [self.reply] * len(batch)

class VLLM:
    def __init__(self, model, max_len=12288, tp=1, gpu_util=0.90, dtype='half', max_tokens=256):
        from vllm import LLM, SamplingParams
        self.llm = LLM(model=model, dtype=dtype, max_model_len=max_len, tensor_parallel_size=tp, gpu_memory_utilization=gpu_util, enable_prefix_caching=True, seed=0)
        self.sp = SamplingParams(temperature=0.0, max_tokens=max_tokens)
    def generate(self, batch):
        outs = self.llm.chat(batch, self.sp, use_tqdm=False, chat_template_kwargs={'enable_thinking': False})
        return [o.outputs[0].text for o in outs]

class HF:
    def __init__(self, model, max_tokens=200, bs=4):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.tok = AutoTokenizer.from_pretrained(model); self.tok.padding_side = 'left'
        self.m = AutoModelForCausalLM.from_pretrained(model, torch_dtype=torch.float16, device_map='auto').eval()
        self.mt, self.bs, self.torch = max_tokens, bs, torch
    def generate(self, batch):
        out = []
        for i in range(0, len(batch), self.bs):
            chunk = batch[i:i + self.bs]
            txt = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in chunk]
            enc = self.tok(txt, return_tensors='pt', padding=True).to(self.m.device)
            with self.torch.no_grad(): g = self.m.generate(**enc, max_new_tokens=self.mt, do_sample=False)
            out += [self.tok.decode(x[enc['input_ids'].shape[1]:], skip_special_tokens=True) for x in g]
        return out

def make(name, model, **kw):
    if name == 'stub': return Stub()
    if name == 'vllm': return VLLM(model, **kw)
    return HF(model)


In [ ]:
%%writefile /kaggle/working/lifemate/common.py
"""Shared helpers for the LifeMate baseline. Pure stdlib (+ optional numpy)."""
import json, sys, re, math, unicodedata, collections
from pathlib import Path

def find_root(hint=None):
    """Folder that holds train/train.jsonl and public_test/public_test.jsonl (works locally and on Kaggle)."""
    cands = [Path(hint)] if hint else []
    cands += [Path('.'), Path('data/lifemate/train+public_test'), Path('/kaggle/input')]
    for base in cands:
        if not base.exists(): continue
        for p in [base] + list(base.glob('*')) + list(base.glob('*/*')) + list(base.glob('*/*/*')):
            if (p / 'train' / 'train.jsonl').is_file(): return p
    raise FileNotFoundError('cannot find train/train.jsonl; pass --root')

def jl(path):
    with open(path, encoding='utf-8') as f: return [json.loads(x) for x in f if x.strip()]

def add_zaic(root, split):
    """Make the split's zaic_tools importable (each split ships its own version)."""
    for k in [k for k in sys.modules if k == 'zaic_tools' or k.startswith('zaic_tools.')]: del sys.modules[k]
    sys.path = [p for p in sys.path if not p.endswith(('/train', '/public_test', '/private_test'))]
    sys.path.insert(0, str(Path(root) / split))
    import zaic_tools
    return zaic_tools

def fold(s):
    t = unicodedata.normalize('NFD', str(s).lower()).replace('đ', 'd')
    return ' '.join(''.join(c for c in t if not unicodedata.combining(c)).split())

# ---------- tiny TF-IDF retriever (char 3-grams over folded text) ----------
class Retriever:
    def __init__(self, docs):
        self.docs = docs
        self.vecs = [self._tf(d) for d in docs]
        df = collections.Counter(k for v in self.vecs for k in v)
        n = len(docs)
        self.idf = {k: math.log((1 + n) / (1 + c)) + 1 for k, c in df.items()}
        self.vecs = [self._norm(v) for v in self.vecs]
    @staticmethod
    def _grams(t):
        t = ' ' + fold(t) + ' '
        return [t[i:i + 3] for i in range(len(t) - 2)]
    def _tf(self, t): return collections.Counter(self._grams(t))
    def _norm(self, v):
        w = {k: c * getattr(self, 'idf', {}).get(k, 1.0) for k, c in v.items()}
        nrm = math.sqrt(sum(x * x for x in w.values())) or 1.0
        return {k: x / nrm for k, x in w.items()}
    def top(self, q, k=4, exclude=None):
        qv = self._norm(self._tf(q)); sc = []
        for i, v in enumerate(self.vecs):
            if exclude is not None and i in exclude: continue
            s = sum(x * v.get(g, 0.0) for g, x in qv.items()); sc.append((s, i))
        sc.sort(reverse=True)
        return [i for _, i in sc[:k]]


In [ ]:
%%writefile /kaggle/working/lifemate/run.py
#!/usr/bin/env python3
"""LifeMate baseline driver.
  python -m lifemate.run asr    --split train --n 300            # transcribe holdout (+history) -> outputs/asr_train.json
  python -m lifemate.run agent  --split train --n 300 --mode oracle|asr --shots 4 --backend vllm
  python -m lifemate.run agent  --split public_test --mode asr --shots 4  # writes outputs/submission_public_test.jsonl
Dev holdout = fixed random sample of train (seed 0); retrieval pool = train minus holdout."""
import argparse, json, random, sys, time, os
from pathlib import Path
from .common import find_root, jl, Retriever
from . import agent as A, score as S

def holdout(rows, n, seed=0):
    ids = [r['id'] for r in rows]; random.Random(seed).shuffle(ids); return set(ids[:n])

def main():
    ap = argparse.ArgumentParser(); ap.add_argument('cmd', choices=['asr', 'agent', 'score_gold'])
    ap.add_argument('--root'); ap.add_argument('--split', default='train'); ap.add_argument('--n', type=int, default=300)
    ap.add_argument('--mode', default='oracle', choices=['oracle', 'asr']); ap.add_argument('--shots', type=int, default=0)
    ap.add_argument('--backend', default='stub'); ap.add_argument('--model', default='Qwen/Qwen3-4B-Instruct-2507')
    ap.add_argument('--asr_model', default='openai/whisper-large-v3-turbo'); ap.add_argument('--out', default='outputs')
    ap.add_argument('--max_len', type=int, default=12288); ap.add_argument('--tp', type=int, default=1); ap.add_argument('--tag', default='')
    a = ap.parse_args(); root = find_root(a.root); out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    train = jl(root / 'train' / 'train.jsonl'); split = a.split
    rows = jl(root / split / f'{split}.jsonl'); cat = json.load(open(root / split / f'{split}_tools_catalog.json', encoding='utf-8'))
    ho = holdout(train, a.n) if split == 'train' else None
    items = [r for r in rows if ho is None or r['id'] in ho]
    if a.cmd == 'asr':
        from .asr import collect, transcribe
        paths = collect(root, split, items); print(len(paths), 'wav files')
        transcribe(root, split, paths, a.asr_model, out=str(out / f'asr_{split}.json')); return
    if a.cmd == 'score_gold':   # sanity: gold calls -> 100, empty -> share of out-of-scope
        zs = S.add_zaic(root, split); subs = {}
        for it in items:
            env = zs.ToolEnv(split, it['id'], root=root)
            for c in it['label']['tool_calls']: env.call(c['name'], c['arguments'])
            subs[it['id']] = {'tool_calls': env.get_tool_calls(), 'state_hash': env.get_state_hash()}
        print('gold ->', S.evaluate(root, split, items, subs, cat)[0])
        zs = S.add_zaic(root, split); empty = {}
        for it in items:
            env = zs.ToolEnv(split, it['id'], root=root); empty[it['id']] = {'tool_calls': [], 'state_hash': env.get_state_hash()}
        print('empty ->', S.evaluate(root, split, items, empty, cat)[0]); return
    # ---- agent ----
    asr_path = out / f'asr_{split}.json'; asr = json.load(open(asr_path, encoding='utf-8')) if asr_path.exists() else {}
    if a.mode == 'asr' and not asr: sys.exit('run the asr stage first')
    def texts_of(it):
        if a.mode == 'oracle' and 'audio_text' in it:
            return it['audio_text'], {h['audio']: h.get('audio_text', '') for h in it.get('history') or [] if h['role'] == 'user'}
        return asr.get(it['audio'], ''), {h['audio']: asr.get(h['audio'], '') for h in it.get('history') or [] if h['role'] == 'user'}
    pool = [r for r in train if (ho is None or r['id'] not in ho)]
    retr = Retriever([r['audio_text'] for r in pool]) if a.shots else None
    kw = {'max_len': a.max_len, 'tp': a.tp} if a.backend == 'vllm' else {}
    from .backends import make; be = make(a.backend, a.model, **kw)
    t0 = time.time(); res = A.run_batch(items, root, split, cat, be, texts_of, pool, retr, a.shots, exclude=False)
    dt = time.time() - t0; tag = a.tag or f'{split}_{a.mode}_s{a.shots}'
    json.dump(res, open(out / f'pred_{tag}.json', 'w', encoding='utf-8'), ensure_ascii=False)
    print(f'{len(items)} samples in {dt:.0f}s ({dt / len(items):.2f}s/sample amortised); max per-sample steps {max(r["steps"] for r in res.values())}')
    if split == 'train':
        sc, per, br = S.evaluate(root, split, items, res, cat); print(f'SCORE {sc:.2f}  breakdown(n, score): {br}')
        json.dump({'score': sc, 'breakdown': br, 'per': per, 'n': len(items), 'seconds': dt}, open(out / f'score_{tag}.json', 'w'), ensure_ascii=False)
    else:
        with open(out / f'submission_{split}.jsonl', 'w', encoding='utf-8') as f:
            for it in items: f.write(json.dumps({'id': it['id'], 'tool_calls': res[it['id']]['tool_calls'], 'state_hash': res[it['id']]['state_hash']}, ensure_ascii=False) + '\n')
        print('wrote', out / f'submission_{split}.jsonl')
if __name__ == '__main__': main()


In [ ]:
%%writefile /kaggle/working/lifemate/score.py
"""Local re-implementation of the official LifeMate metric (from the portal's Evaluation tab).
Sample score: gold empty -> 1 iff submission empty else 0; state_hash != gold or M==0 -> 0; else F1 with P=M/(M+U), R=M/|G|.
A call matches a gold call if same tool name, every non-fuzzy arg identical, every fuzzy arg token-F1 > 0.6 (lower+NFC).
Each call matches at most one; order ignored. Rejected (schema-invalid) calls count as unmatched. Final = 100/N * sum."""
import json, unicodedata, re, collections
from .common import add_zaic

def _nfc(s): return unicodedata.normalize('NFC', str(s)).lower()
def _toks(s): return re.findall(r'\w+', _nfc(s))
def token_f1(a, b):
    ta, tb = _toks(a), _toks(b)
    if not ta and not tb: return 1.0
    common = sum((collections.Counter(ta) & collections.Counter(tb)).values())
    if common == 0: return 0.0
    p, r = common / len(ta), common / len(tb); return 2 * p * r / (p + r)

def fuzzy_keys(catalog):
    out = {}
    for a in catalog['apps']:
        for t in a['tools']:
            props = (t.get('parameters') or {}).get('properties', {})
            out[t['name']] = {k for k, v in props.items() if v.get('x-match') == 'fuzzy'}
    return out

def _same(a, b):
    if isinstance(a, str) and isinstance(b, str): return ' '.join(a.split()) == ' '.join(b.split())
    return json.dumps(a, sort_keys=True, ensure_ascii=False) == json.dumps(b, sort_keys=True, ensure_ascii=False)

def calls_match(sub, gold, fz):
    if sub['name'] != gold['name']: return False
    sa, ga = sub.get('arguments') or {}, gold.get('arguments') or {}
    f = fz.get(gold['name'], set())
    for k in set(sa) | set(ga):
        if k in f:
            if k not in sa or k not in ga: return False
            if token_f1(sa[k], ga[k]) <= 0.6: return False
        elif k not in sa or k not in ga or not _same(sa[k], ga[k]): return False
    return True

def sample_score(sub_calls, sub_hash, gold_calls, gold_hash, fz):
    if not gold_calls: return 1.0 if not sub_calls else 0.0
    if sub_hash != gold_hash: return 0.0
    used, M = set(), 0
    for s in sub_calls:
        for j, g in enumerate(gold_calls):
            if j not in used and calls_match(s, g, fz): used.add(j); M += 1; break
    if M == 0: return 0.0
    U = len(sub_calls) - M; P, R = M / (M + U), M / len(gold_calls)
    return 2 * P * R / (P + R)

def gold_hashes(root, split, rows):
    """state_hash of the gold final state, obtained by replaying the gold calls in a fresh ToolEnv."""
    z = add_zaic(root, split); out = {}
    for it in rows:
        env = z.ToolEnv(split, it['id'], root=root)
        for c in it['label']['tool_calls']: env.call(c['name'], c['arguments'])
        out[it['id']] = env.get_state_hash()
    return out

def evaluate(root, split, rows, subs, catalog):
    """rows: train rows (with label); subs: {id: {'tool_calls', 'state_hash'}}. Returns (final_score, per-id scores, breakdown)."""
    fz = fuzzy_keys(catalog); gh = gold_hashes(root, split, rows); per = {}
    for it in rows:
        s = subs.get(it['id'], {'tool_calls': [], 'state_hash': None})
        per[it['id']] = sample_score(s['tool_calls'], s['state_hash'], it['label']['tool_calls'], gh[it['id']], fz)
    groups = collections.defaultdict(list)
    for it in rows:
        g = it['label']['tool_calls']
        kind = 'out_of_scope' if not g else ('ask' if g[-1]['name'] == 'sys.ask_missing_info' else 'execute')
        groups[kind].append(per[it['id']])
    n = len(rows)
    return 100 * sum(per.values()) / n, per, {k: (len(v), 100 * sum(v) / len(v)) for k, v in groups.items()}


In [ ]:
!cd /kaggle/working && python -m lifemate.run asr --split train --n 300 --asr_model openai/whisper-large-v3-turbo --out outputs

In [ ]:
!cd /kaggle/working && python -m lifemate.run agent --split train --n 300 --mode oracle --shots 0 --backend vllm --model Qwen/Qwen3-4B-Instruct-2507 --out outputs --tag z0
!cd /kaggle/working && python -m lifemate.run agent --split train --n 300 --mode oracle --shots 4 --backend vllm --model Qwen/Qwen3-4B-Instruct-2507 --out outputs --tag s4
!cd /kaggle/working && python -m lifemate.run agent --split train --n 300 --mode asr --shots 4 --backend vllm --model Qwen/Qwen3-4B-Instruct-2507 --out outputs --tag asr4